In [19]:
from typing import Annotated, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from typing import TypedDict
from langgraph.graph import START, END, StateGraph
from langchain_groq import ChatGroq

class llmResponse(BaseModel):
    sentiment: Literal[1, -1] = Field(
        description="1 for positive sentiment, -1 for negative sentiment"
    )

class DiagnosisResponse(BaseModel):
    issueType:str = Field(
        description="the issue type"
    )
    tone:str = Field(
        description="the tone of customer"
    )
    urgency:str = Field(
        description="the urgency of customer"
    )

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)
structured_llm=llm.with_structured_output(llmResponse)
structured_llm_diagnosis=llm.with_structured_output(DiagnosisResponse)
# result=structured_llm.invoke("this is a good review")
# print(result.sentiment)


class ReviewAnalyser(TypedDict):
    review:str
    sentiment:Literal[1,-1]
    issue_type:str
    tone:str
    urgency:str
    response:str





def findSentiment(state:ReviewAnalyser)->dict:
    result=structured_llm.invoke(state["review"])

    return {
        "sentiment":result.sentiment
    }

def checkCentiment(state:ReviewAnalyser)->Literal["positiveResponse","runDiagnosis"]:
    if state["sentiment"]==1:
        return 'positiveResponse'
    else:
        return 'runDiagnosis'
    
def runDiagnosis(state:ReviewAnalyser)->dict:
    result=structured_llm_diagnosis.invoke(state["review"])

    return {
        "issue_type":result.issueType,
        'tone':result.tone,
        'urgency':result.urgency
    }
def negativeResponse(state: ReviewAnalyser) -> dict:
    prompt = f"""
    You are a professional customer support executive.

    A customer has left a NEGATIVE review.

    Customer review:
    {state['review']}

    Extracted analysis:
    - Issue type: {state['issue_type']}
    - Customer tone: {state['tone']}
    - Urgency level: {state['urgency']}

    Write a response that:
    1. Apologizes sincerely.
    2. Acknowledges the specific issue type.
    3. Matches the customer's tone appropriately (calm and empathetic).
    4. Addresses the urgency level.
    5. Offers a clear next step or resolution.
    6. Keeps the response between 60 and 120 words.

    Do not blame the customer.
    Return only the response text.
    """

    result = llm.invoke(prompt)

    return {
        "response": result.content
    }


def positiveResponse(state: ReviewAnalyser) -> dict:
    prompt = f"""
    You are a friendly and professional customer support representative.

    A customer has left a POSITIVE review:

    {state['review']}

    Write a warm thank-you response that:
    - Appreciates the customer's feedback
    - Acknowledges their positive experience
    - Sounds genuine and professional
    - Encourages them to continue using the product or service
    - Keeps the response between 40 and 80 words

    Return only the response text.
    """

    result = llm.invoke(prompt)

    return {
        "response": result.content
    }

graph = StateGraph(ReviewAnalyser)

graph.add_node("findSentiment", findSentiment)
graph.add_node("runDiagnosis", runDiagnosis)
graph.add_node("negativeResponse", negativeResponse)
graph.add_node("positiveResponse", positiveResponse)


graph.add_edge(START,'findSentiment')

graph.add_conditional_edges('findSentiment',checkCentiment)

graph.add_edge('runDiagnosis','negativeResponse')

graph.add_edge('negativeResponse',END)
graph.add_edge('positiveResponse',END)


workflow = graph.compile()

review8 = "I was charged twice for the same subscription and I need this corrected immediately. This billing error is causing serious inconvenience."
initial_state={
    "review":review8
}

final_state=workflow.invoke(initial_state)

filtered_state = {k: v for k, v in final_state.items() if k != "review"}

print(filtered_state)



{'sentiment': -1, 'issue_type': 'billing error', 'tone': 'angry', 'urgency': 'high', 'response': "I apologize sincerely for the billing error that resulted in a duplicate charge. I understand how frustrating this must be for you. I'm here to help resolve this issue as quickly as possible. I'll expedite the correction of the error and ensure a refund is processed promptly. Please allow me to investigate further and I will be in touch with a resolution within the next 24 hours."}


{'sentiment': -1, 'issue_type': 'billing error', 'tone': 'angry', 'urgency': 'high', 'response': "I apologize sincerely for the billing error that resulted in a duplicate charge. I understand how frustrating this must be for you. I'm here to help resolve this issue as quickly as possible. I'll expedite the correction of the error and ensure a refund is processed promptly. Please allow me to investigate further and I will be in touch with a resolution within the next 24 hours."}

In [ ]:
{'sentiment': llmResponse(sentiment=1), 'issue_type': 'None', 'tone': 'Positive', 'urgency': 'Low', 'response': "I apologize, but it seems there's been a misunderstanding. Since there's no issue with your laptop purchase, I'd like to simply thank you for your positive review. We're thrilled you're satisfied with the performance. If you have any questions or need assistance in the future, please don't hesitate to reach out."}